# Experiment 5.5.1 — Regularized Semantic WHEN

**Analysis-only notebook.** The validation selector and finalizer are external experiment stages. This notebook only reads finalized CSV/JSON artifacts; missing artifacts are treated as an incomplete experiment and are never regenerated here.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')

repo_root = find_repo_root()
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_5_5_1_regularized_semantic_when' / 'regularized_semantic_when_v1'
required = [
    'screen_runs.csv', 'screen_summary.csv', 'screen_paired_deltas.csv', 'selection.json',
    'final_runs.csv', 'final_summary.csv', 'final_paired_deltas.csv',
    'progress_metrics.csv', 'state_diagnostics.csv', 'state_profiles.csv',
    'q_only_probes.csv', 'q_progress_probe.csv', 'shuffled_q.csv',
    'same_what_pairs.csv', 'matched_progress_same_what.csv', 'manifest.json',
]
missing = [name for name in required if not (root / name).exists()]
if missing:
    raise FileNotFoundError(f'Exp5.5.1 is not finalized; missing artifacts: {missing}')

screen_runs = pd.read_csv(root / 'screen_runs.csv')
screen_summary = pd.read_csv(root / 'screen_summary.csv')
screen_paired = pd.read_csv(root / 'screen_paired_deltas.csv')
selection = json.loads((root / 'selection.json').read_text())
final_runs = pd.read_csv(root / 'final_runs.csv')
final_summary = pd.read_csv(root / 'final_summary.csv')
final_paired = pd.read_csv(root / 'final_paired_deltas.csv')
progress = pd.read_csv(root / 'progress_metrics.csv')
state_diag = pd.read_csv(root / 'state_diagnostics.csv')
state_profiles = pd.read_csv(root / 'state_profiles.csv')
q_only = pd.read_csv(root / 'q_only_probes.csv')
q_progress = pd.read_csv(root / 'q_progress_probe.csv')
shuffled = pd.read_csv(root / 'shuffled_q.csv')
same_what = pd.read_csv(root / 'same_what_pairs.csv')
matched_progress_same_what = pd.read_csv(root / 'matched_progress_same_what.csv')
manifest = json.loads((root / 'manifest.json').read_text())
root

## Validation-only recipe screen

In [ ]:
display(pd.DataFrame([selection]))
display(screen_summary.sort_values('complexity'))
display(screen_paired.groupby('recipe')['delta_val_balanced_accuracy_vs_ce_only'].agg(['mean', 'std', 'min', 'max']))

In [ ]:
order = ['ce_only', 'progress', 'progress_sticky', 'progress_sticky_confidence']
pivot = screen_runs.pivot(index='seed', columns='recipe', values='val_balanced_accuracy')
fig, ax = plt.subplots(figsize=(9, 5))
for seed, row in pivot.iterrows():
    x = [name for name in order if name in row.index]
    ax.plot(x, [row[name] for name in x], marker='o', alpha=0.65, label=f'seed {seed}')
ax.set_ylabel('Validation balanced accuracy')
ax.set_title('Exp5.5.1 validation-only nested regularization screen')
ax.tick_params(axis='x', rotation=20)
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()

## Final classification results

In [ ]:
display(final_summary.sort_values('mean_test_balanced_accuracy', ascending=False))
display(final_paired.groupby('comparison')['delta_test_balanced_accuracy'].agg(['mean', 'std', 'min', 'max']))

In [ ]:
plot_order = ['shared', 'ce_reset', 'ce_ordered', 'reg_reset', 'reg_ordered', 'clock', 'oracle_progress', 'fixed250', 'reg_ordered_shuffled_q']
plot_frame = final_summary.set_index('condition').reindex(plot_order).dropna(how='all')
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(np.arange(len(plot_frame)), plot_frame['mean_test_balanced_accuracy'])
ax.errorbar(np.arange(len(plot_frame)), plot_frame['mean_test_balanced_accuracy'], yerr=plot_frame['sem_test_balanced_accuracy'], fmt='none', capsize=3)
ax.set_xticks(np.arange(len(plot_frame)), plot_frame.index, rotation=30, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Final paired Exp5.5 / Exp5.5.1 comparison')
plt.tight_layout()

## Progress auxiliary diagnostics

In [ ]:
display(progress[progress['split'] == 'test'].sort_values(['condition', 'seed']))
display(progress[progress['split'] == 'test'].groupby('condition')[['mae', 'rmse', 'r2', 'spearman', 'coarse_10bin_accuracy']].mean())

## Semantic state geometry and persistence

In [ ]:
key_metrics = ['entropy_mean', 'total_variation_l1_mean', 'argmax_transitions_per_second', 'mean_dwell_seconds']
display(state_diag[state_diag['metric'].isin(key_metrics)].pivot_table(index='condition', columns='metric', values='value', aggfunc='mean'))

In [ ]:
profile = state_profiles[(state_profiles['condition'] == 'reg_ordered') & (state_profiles['axis'] == 'relative_progress')]
profile_mean = profile.groupby(['position', 'state'], as_index=False)['probability'].mean()
fig, ax = plt.subplots(figsize=(9, 5))
for state, group in profile_mean.groupby('state'):
    ax.plot(group['position'], group['probability'], marker='o', label=f'state {int(state)}')
ax.set_xlabel('Relative progress')
ax.set_ylabel('Mean q probability')
ax.set_title('Selected regularized ordered q profile')
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()

## Class-code and progress-code checks

In [ ]:
display(q_only[q_only['split'] == 'test'].pivot_table(index=['condition', 'seed'], columns='feature', values='balanced_accuracy'))
display(q_progress.sort_values(['condition', 'seed']))

## Same-WHAT / different-history mechanism checks

In [ ]:
print('same-WHAT rows:', len(same_what))
print('same-WHAT + matched-progress rows:', len(matched_progress_same_what))
if not same_what.empty:
    display(same_what.sort_values(['what_cosine_similarity', 'q_l1_distance'], ascending=False).head(30))
if not matched_progress_same_what.empty:
    display(matched_progress_same_what.sort_values(['what_cosine_similarity', 'q_l1_distance'], ascending=False).head(30))

## Interpretation checklist

Primary rescue requires `reg_ordered > ce_ordered`, `reg_ordered > reg_reset`, and `reg_ordered > reg_ordered_shuffled_q`. Clock/oracle comparisons determine whether the learned WHEN is merely an internal phase estimator or captures additional history-conditioned structure. The progress-to-q Ridge diagnostic and matched-progress same-WHAT table should be consulted before describing q as semantic state rather than a progress code.